# Education-level subset inspection

Filter `data/processed/ready.jsonl` to a chosen set of education stages
(High school / Middle school / Primary school / Preparatory / Licence) and inspect
everything: language, subject, question type, level overlap, search-text length,
duplicates, authors, samples.

Toggle stages on/off in the **setup** cell. Default = high school only.

Source: `data/processed/ready.jsonl` (Stage-2 normalized output).
Output:  `data/interim/levels_subset_<TAG>.jsonl` (filtered subset, tag derived from selection).

## 0. Setup — choose which education stages to include

Five stages exist in the corpus. Set each `INCLUDE_*` flag to `True` to pull that
stage into the subset. The level lists per stage are exhaustive (every level present
in `ready.jsonl`).

In [50]:
from pathlib import Path
import sys, json
from collections import Counter, defaultdict

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

READY_PATH = ROOT / "data/processed/ready.jsonl"
OUT_DIR = ROOT / "data/interim"

# ---- Level catalog grouped by education stage ----
LEVEL_CATEGORIES = {
    "HIGH_SCHOOL": [
        "HIGH_SCHOOL_1ST_GRADE",
        "HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE",
        "HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE",
        "HIGH_SCHOOL_2ND_GRADE_LETTRES",
        "HIGH_SCHOOL_2ND_GRADE_SCIENCE",
        "HIGH_SCHOOL_3RD_GRADE_COMPUTER_SCIENCE",
        "HIGH_SCHOOL_3RD_GRADE_ECONOMICS",
        "HIGH_SCHOOL_3RD_GRADE_LETTRES",
        "HIGH_SCHOOL_3RD_GRADE_MATHEMATICS",
        "HIGH_SCHOOL_3RD_GRADE_SCIENCE",
        "HIGH_SCHOOL_3RD_GRADE_TECHNIQUES",
        "HIGH_SCHOOL_4TH_GRADE_COMPUTER_SCIENCE",
        "HIGH_SCHOOL_4TH_GRADE_ECONOMICS",
        "HIGH_SCHOOL_4TH_GRADE_LETTRES",
        "HIGH_SCHOOL_4TH_GRADE_MATHEMATICS",
        "HIGH_SCHOOL_4TH_GRADE_SCIENCE",
        "HIGH_SCHOOL_4TH_GRADE_TECHNIQUES",
    ],
    "MIDDLE_SCHOOL": [
        "MIDDLE_SCHOOL_1ST_GRADE",
        "MIDDLE_SCHOOL_2ND_GRADE",
        "MIDDLE_SCHOOL_3RD_GRADE",
    ],
    "PRIMARY_SCHOOL": [
        "PRIMARY_SCHOOL_1ST_GRADE",
        "PRIMARY_SCHOOL_2ND_GRADE",
        "PRIMARY_SCHOOL_3RD_GRADE",
        "PRIMARY_SCHOOL_4TH_GRADE",
        "PRIMARY_SCHOOL_5TH_GRADE",
        "PRIMARY_SCHOOL_6TH_GRADE",
    ],
    "PREPARATORY": [
        "PREPARATORY_1ST_BG",
        "PREPARATORY_1ST_MP",
        "PREPARATORY_1ST_MPI",
        "PREPARATORY_1ST_PC",
        "PREPARATORY_1ST_TECHNO",
        "PREPARATORY_2ND_BG",
        "PREPARATORY_2ND_MP",
        "PREPARATORY_2ND_PC",
        "PREPARATORY_2ND_TECHNO",
    ],
    "LICENCE": [
        "LICENCE_1ST_GRADE",
        "LICENCE_2ND_GRADE",
        "LICENCE_3RD_GRADE",
    ],
}

# ---- Toggle which stages to include in the subset ----
INCLUDE_HIGH_SCHOOL = True
INCLUDE_MIDDLE_SCHOOL = False
INCLUDE_PRIMARY_SCHOOL = False
INCLUDE_PREPARATORY = False
INCLUDE_LICENCE = False

STAGE_FLAGS = {
    "HIGH_SCHOOL": INCLUDE_HIGH_SCHOOL,
    "MIDDLE_SCHOOL": INCLUDE_MIDDLE_SCHOOL,
    "PRIMARY_SCHOOL": INCLUDE_PRIMARY_SCHOOL,
    "PREPARATORY": INCLUDE_PREPARATORY,
    "LICENCE": INCLUDE_LICENCE,
}
SELECTED_STAGES = [s for s, on in STAGE_FLAGS.items() if on]
if not SELECTED_STAGES:
    raise ValueError("No stages selected — set at least one INCLUDE_* flag to True.")

TARGET_LEVELS = [lv for s in SELECTED_STAGES for lv in LEVEL_CATEGORIES[s]]
TARGET_SET = set(TARGET_LEVELS)
LEVEL_TO_STAGE = {lv: s for s, lvs in LEVEL_CATEGORIES.items() for lv in lvs}

OUT_TAG = "_".join(SELECTED_STAGES)
OUT_PATH = OUT_DIR / f"levels_subset_{OUT_TAG}.jsonl"

print("Selected stages:", SELECTED_STAGES)
print(f"Target levels:   {len(TARGET_LEVELS)}")
print(f"Source:          {READY_PATH}")
print(f"Output will go:  {OUT_PATH}")

Selected stages: ['HIGH_SCHOOL']
Target levels:   17
Source:          /Users/macmini/Documents/work/softy/quiz-generator/data/processed/ready.jsonl
Output will go:  /Users/macmini/Documents/work/softy/quiz-generator/data/interim/levels_subset_HIGH_SCHOOL.jsonl


## 1. Load and filter

Keep any row whose `levels` list contains at least one target level. Rows can carry
multiple levels, so the same row may match more than one target.

In [51]:
rows = []
total = 0
with READY_PATH.open() as f:
    for line in f:
        total += 1
        rec = json.loads(line)
        if TARGET_SET.intersection(rec.get("levels") or []):
            rows.append(rec)

df = pd.DataFrame(rows)
print(f"Total rows in source: {total}")
print(f"Filtered rows: {len(df)} ({len(df)/total:.1%})")
df.head(3)

Total rows in source: 10287
Filtered rows: 3918 (38.1%)


,doc_id,quiz_id,quiz_title,language,subjects,levels,question_type,multiple_correct_answers,question_text,choices_text,correct_choices_text,choices_media,points,time,author_name,author_email,search_text
0,655f1adf30d4e850205b75bb__q0,655f1adf30d4e850205b75bb,النّظام الغذائي صحّيّ,ar,[],"[PRIMARY_SCHOOL_4TH_GRADE, PRIMARY_SCHOOL_5TH_...",MULTIPLE_CHOICE,False,ما هو النظام الغذائي الصحي؟,[نظام غذائي متوازن يشمل جميع العناصر الغذائية ...,[كُل ما ذُكِرَ],"[None, None, None, None]",1.0,30,Amira Abdelhamid,amiraabdelhamid@takiacademyteam.com,النّظام الغذائي صحّيّ. ما هو النظام الغذائي ال...
1,655f1adf30d4e850205b75bb__q1,655f1adf30d4e850205b75bb,النّظام الغذائي صحّيّ,ar,[],"[PRIMARY_SCHOOL_4TH_GRADE, PRIMARY_SCHOOL_5TH_...",MULTIPLE_CHOICE,True,ما أهمية إتباع نظام غذائي صحي ؟,[يُعزز من الجهاز المناعي للجسم والوقاية من الأ...,[يُعزز من الجهاز المناعي للجسم والوقاية من الأ...,"[None, None, None, None, None]",1.0,30,Amira Abdelhamid,amiraabdelhamid@takiacademyteam.com,النّظام الغذائي صحّيّ. ما أهمية إتباع نظام غذا...
2,655f1adf30d4e850205b75bb__q3,655f1adf30d4e850205b75bb,النّظام الغذائي صحّيّ,ar,[],"[PRIMARY_SCHOOL_4TH_GRADE, PRIMARY_SCHOOL_5TH_...",MULTIPLE_CHOICE,False,كم عدد الخضروات و الغلال المنصوح تناولها في ال...,"[2, 10, 8, 5]",[5],"[None, None, None, None]",1.0,30,Amira Abdelhamid,amiraabdelhamid@takiacademyteam.com,النّظام الغذائي صحّيّ. كم عدد الخضروات و الغلا...


In [52]:
print("Columns:", list(df.columns))
print("\ndtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isna().sum())

Columns: ['doc_id', 'quiz_id', 'quiz_title', 'language', 'subjects', 'levels', 'question_type', 'multiple_correct_answers', 'question_text', 'choices_text', 'correct_choices_text', 'choices_media', 'points', 'time', 'author_name', 'author_email', 'search_text']

dtypes:
doc_id                       object
quiz_id                      object
quiz_title                   object
language                     object
subjects                     object
levels                       object
question_type                object
multiple_correct_answers       bool
question_text                object
choices_text                 object
correct_choices_text         object
choices_media                object
points                      float64
time                          int64
author_name                  object
author_email                 object
search_text                  object
dtype: object

Null counts:
doc_id                      0
quiz_id                     0
quiz_title                  0

## 2. Per-stage and per-level row count

How many rows touch each selected stage and each individual level. Sum across levels
is greater than the filtered total because rows can be tagged with multiple levels.

In [53]:
stage_counts = Counter()
level_counts = Counter()
for levels in df["levels"]:
    seen_stages = set()
    for lv in levels:
        if lv in TARGET_SET:
            level_counts[lv] += 1
            seen_stages.add(LEVEL_TO_STAGE[lv])
    for s in seen_stages:
        stage_counts[s] += 1

print("Rows per selected stage (a row can fall into multiple stages):")
for s in SELECTED_STAGES:
    print(f"  {s:15s} {stage_counts.get(s, 0)}")

level_df = (
    pd.DataFrame(
        [(LEVEL_TO_STAGE[lv], lv, level_counts.get(lv, 0)) for lv in TARGET_LEVELS],
        columns=["stage", "level", "row_count"],
    )
    .sort_values(["stage", "row_count"], ascending=[True, False])
    .reset_index(drop=True)
)
print(f"\nSum of per-level rows: {level_df['row_count'].sum()} (greater than {len(df)} due to multi-level rows)")
level_df

Rows per selected stage (a row can fall into multiple stages):
  HIGH_SCHOOL     3918

Sum of per-level rows: 10765 (greater than 3918 due to multi-level rows)


,stage,level,row_count
0,HIGH_SCHOOL,HIGH_SCHOOL_2ND_GRADE_SCIENCE,1125
1,HIGH_SCHOOL,HIGH_SCHOOL_4TH_GRADE_MATHEMATICS,1041
2,HIGH_SCHOOL,HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE,1036
3,HIGH_SCHOOL,HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE,1002
4,HIGH_SCHOOL,HIGH_SCHOOL_4TH_GRADE_TECHNIQUES,953
5,HIGH_SCHOOL,HIGH_SCHOOL_4TH_GRADE_SCIENCE,931
6,HIGH_SCHOOL,HIGH_SCHOOL_4TH_GRADE_COMPUTER_SCIENCE,856
7,HIGH_SCHOOL,HIGH_SCHOOL_1ST_GRADE,801
8,HIGH_SCHOOL,HIGH_SCHOOL_4TH_GRADE_ECONOMICS,508
9,HIGH_SCHOOL,HIGH_SCHOOL_3RD_GRADE_MATHEMATICS,457


## 3. How many levels does each row carry?

If most rows are tagged with a single level, the dataset is well partitioned. If many
rows carry 10+ levels, the metadata is fuzzy and per-level retrieval will overlap a lot.

In [54]:
df["n_levels_total"] = df["levels"].map(len)
df["n_levels_in_target"] = df["levels"].map(lambda lvs: sum(1 for x in lvs if x in TARGET_SET))

print("Total levels per row (count of all level tags):")
print(df["n_levels_total"].describe())
print("\nTarget levels per row (count of selected-stage tags):")
print(df["n_levels_in_target"].describe())

print("\nRow distribution by total-levels-tagged:")
print(df["n_levels_total"].value_counts().sort_index())

Total levels per row (count of all level tags):
count    3918.000000
mean        2.867024
std         3.240970
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        38.000000
Name: n_levels_total, dtype: float64

Target levels per row (count of selected-stage tags):
count    3918.000000
mean        2.747575
std         2.464522
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        17.000000
Name: n_levels_in_target, dtype: float64

Row distribution by total-levels-tagged:
n_levels_total
1     1743
2      241
3      721
4      710
5      239
6      186
7       10
23      64
38       4
Name: count, dtype: int64


## 4. Language distribution

Quizzes can be Arabic, French or English. Note the field is the quiz-declared language,
not detected — Stage 2 normalization keeps the source value.

In [55]:
lang_counts = df["language"].fillna("<missing>").value_counts()
print("Language distribution:")
print(lang_counts)
print(f"\nShare:")
print((lang_counts / len(df)).round(3))

Language distribution:
language
fr    1895
en    1403
ar     620
Name: count, dtype: int64

Share:
language
fr    0.484
en    0.358
ar    0.158
Name: count, dtype: float64


## 5. Subject distribution

`subjects` is a list (a quiz can be tagged with multiple). Count rows per subject and
track rows that carry no subject at all — those are unsearchable by subject filter.

In [56]:
subj_counts = Counter()
no_subject = 0
for subjects in df["subjects"]:
    if not subjects:
        no_subject += 1
        continue
    for s in subjects:
        subj_counts[s] += 1

print(f"Rows with no subject: {no_subject} ({no_subject/len(df):.1%})")
print(f"Distinct subjects: {len(subj_counts)}")
subj_df = (
    pd.DataFrame(subj_counts.most_common(), columns=["subject", "row_count"])
    .reset_index(drop=True)
)
subj_df

Rows with no subject: 92 (2.3%)
Distinct subjects: 10


,subject,row_count
0,ENGLISH,1372
1,MATHEMATICS,1003
2,ARABIC,519
3,PHYSICS,402
4,CHEMISTRY,340
5,SCIENCE,125
6,TECHNIQUE,22
7,HISTORY,20
8,COMPUTER_SCIENCE,13
9,FRENCH,10


## 6. Question type & multiple-correct flag

In [57]:
print("Question type:")
print(df["question_type"].value_counts(dropna=False))
print("\nMultiple correct answers:")
print(df["multiple_correct_answers"].value_counts(dropna=False))
print("\nQuestion type x multiple_correct (cross-tab):")
pd.crosstab(df["question_type"].fillna("<missing>"), df["multiple_correct_answers"].fillna("<missing>"))

Question type:
question_type
MULTIPLE_CHOICE       3881
FILL_IN_THE_BLANKS      37
Name: count, dtype: int64

Multiple correct answers:
multiple_correct_answers
False    3723
True      195
Name: count, dtype: int64

Question type x multiple_correct (cross-tab):


multiple_correct_answers,False,True
question_type,,
FILL_IN_THE_BLANKS,37,0
MULTIPLE_CHOICE,3686,195


## 7. Subject × Level cross-tab

For each target level, what subjects are present? This shows, for example, that
`HIGH_SCHOOL_4TH_GRADE_MATHEMATICS` should be dominated by MATHEMATICS, but other
subjects can leak in via multi-level quizzes.

In [58]:
level_subj = defaultdict(Counter)
for _, row in df.iterrows():
    targets_in_row = [lv for lv in row["levels"] if lv in TARGET_SET]
    subjects = row["subjects"] or ["<no_subject>"]
    for lv in targets_in_row:
        for s in subjects:
            level_subj[lv][s] += 1

all_subjects = sorted({s for c in level_subj.values() for s in c})
matrix = pd.DataFrame(
    {lv: [level_subj[lv].get(s, 0) for s in all_subjects] for lv in TARGET_LEVELS},
    index=all_subjects,
)
matrix.index.name = "subject"
matrix.loc["_TOTAL"] = matrix.sum(axis=0)
matrix

,HIGH_SCHOOL_1ST_GRADE,HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE,HIGH_SCHOOL_2ND_GRADE_LETTRES,HIGH_SCHOOL_2ND_GRADE_SCIENCE,HIGH_SCHOOL_3RD_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_3RD_GRADE_ECONOMICS,HIGH_SCHOOL_3RD_GRADE_LETTRES,HIGH_SCHOOL_3RD_GRADE_MATHEMATICS,HIGH_SCHOOL_3RD_GRADE_SCIENCE,HIGH_SCHOOL_3RD_GRADE_TECHNIQUES,HIGH_SCHOOL_4TH_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_4TH_GRADE_ECONOMICS,HIGH_SCHOOL_4TH_GRADE_LETTRES,HIGH_SCHOOL_4TH_GRADE_MATHEMATICS,HIGH_SCHOOL_4TH_GRADE_SCIENCE,HIGH_SCHOOL_4TH_GRADE_TECHNIQUES
subject,,,,,,,,,,,,,,,,,
<no_subject>,31,41,31,40,41,42,31,39,52,42,42,39,39,39,44,44,44
ARABIC,286,237,222,237,237,4,32,4,4,4,4,4,4,4,4,4,4
CHEMISTRY,0,0,0,0,5,104,0,2,140,116,115,55,0,0,64,64,144
COMPUTER_SCIENCE,7,6,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0
ENGLISH,267,595,609,52,610,0,0,88,0,0,0,397,387,171,387,387,387
FRENCH,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,0,0
HISTORY,0,0,10,0,0,0,0,0,0,0,0,0,0,10,0,0,0
MATHEMATICS,146,90,131,7,148,30,10,0,83,92,72,187,45,0,340,236,152
PHYSICS,31,0,0,0,0,137,0,0,128,134,110,141,0,0,159,133,177


## 8. Language × Level cross-tab

In [59]:
level_lang = defaultdict(Counter)
for _, row in df.iterrows():
    targets_in_row = [lv for lv in row["levels"] if lv in TARGET_SET]
    lang = row["language"] or "<missing>"
    for lv in targets_in_row:
        level_lang[lv][lang] += 1

all_langs = sorted({l for c in level_lang.values() for l in c})
lang_matrix = pd.DataFrame(
    {lv: [level_lang[lv].get(l, 0) for l in all_langs] for lv in TARGET_LEVELS},
    index=all_langs,
)
lang_matrix.index.name = "language"
lang_matrix

,HIGH_SCHOOL_1ST_GRADE,HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE,HIGH_SCHOOL_2ND_GRADE_LETTRES,HIGH_SCHOOL_2ND_GRADE_SCIENCE,HIGH_SCHOOL_3RD_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_3RD_GRADE_ECONOMICS,HIGH_SCHOOL_3RD_GRADE_LETTRES,HIGH_SCHOOL_3RD_GRADE_MATHEMATICS,HIGH_SCHOOL_3RD_GRADE_SCIENCE,HIGH_SCHOOL_3RD_GRADE_TECHNIQUES,HIGH_SCHOOL_4TH_GRADE_COMPUTER_SCIENCE,HIGH_SCHOOL_4TH_GRADE_ECONOMICS,HIGH_SCHOOL_4TH_GRADE_LETTRES,HIGH_SCHOOL_4TH_GRADE_MATHEMATICS,HIGH_SCHOOL_4TH_GRADE_SCIENCE,HIGH_SCHOOL_4TH_GRADE_TECHNIQUES
language,,,,,,,,,,,,,,,,,
ar,357,301,296,301,301,68,96,68,68,68,68,68,68,78,68,78,68
en,267,596,609,61,611,0,0,96,0,0,0,405,395,179,400,400,400
fr,177,105,131,7,213,282,10,2,389,353,318,383,45,0,573,453,485


## 9. Search-text length

Approximate token count (whitespace-split) of `search_text` — this is the field that
gets embedded by BGE-M3 in Stage 3.

In [60]:
df["search_tokens"] = df["search_text"].fillna("").str.split().map(len)
df["question_chars"] = df["question_text"].fillna("").str.len()
df["n_choices"] = df["choices_text"].map(lambda c: len(c) if isinstance(c, list) else 0)

print("search_text token count:")
print(df["search_tokens"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("\nquestion_text char count:")
print(df["question_chars"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("\nNumber of choices:")
print(df["n_choices"].value_counts().sort_index())

search_text token count:
count    3918.000000
mean       29.623022
std        14.212008
min         6.000000
50%        26.000000
90%        46.000000
95%        54.000000
99%        83.000000
max       161.000000
Name: search_tokens, dtype: float64

question_text char count:
count     3918.000000
mean       244.099030
std       2540.727946
min          4.000000
50%         80.000000
90%        187.000000
95%        244.150000
99%        435.660000
max      55977.000000
Name: question_chars, dtype: float64

Number of choices:
n_choices
1      37
2     743
3    2484
4     607
5      27
6      16
7       4
Name: count, dtype: int64


## 10. Quiz / question duplicates within the subset

Stage 2 already deduplicates on `doc_id`, but within the slice we may still find many
questions sharing a `quiz_id` (same quiz) or duplicated `question_text`.

In [61]:
print(f"Distinct doc_id:        {df['doc_id'].nunique()} / {len(df)}")
print(f"Distinct quiz_id:       {df['quiz_id'].nunique()}")
print(f"Distinct quiz_title:    {df['quiz_title'].nunique()}")
print(f"Distinct question_text: {df['question_text'].nunique()}")

qt_dups = df["question_text"].value_counts()
qt_dups = qt_dups[qt_dups > 1]
print(f"\nQuestion texts repeated more than once: {len(qt_dups)}")
print("Top 10 repeated question texts:")
qt_dups.head(10)

Distinct doc_id:        3679 / 3918
Distinct quiz_id:       467
Distinct quiz_title:    442
Distinct question_text: 3828

Question texts repeated more than once: 61
Top 10 repeated question texts:


question_text
C'est le symbole de la porte logique:                                               5
In which sentence is the present perfect used correctly?                            4
Choose the correct option.                                                          4
C'est la table de vérité de la porte logique:                                       4
أيُّ هَذِهِ الأمثِلَةِ لَمْ يُسنَدْ فِيهَا الفِعْلُ إلَى فَاعِلِهِ الحَقِيقِيِّ؟    4
Choose the correct sentence:                                                        4
La molécule ci-dessous se nomme :                                                   4
: C'est la formule brute des ........................                               4
أيُّ هَذِهِ التّشَابِيهِ يُعَدُّ تَشبِيهًا تَمثِيلِيًّا؟                            4
أيُّ هَذِهِ الجُمَلِ تَحْتَوِي عَلى استِثْناءٍ؟                                     3
Name: count, dtype: int64

## 11. Top quizzes and authors

In [62]:
print("Top 15 quizzes by question count:")
top_quizzes = (
    df.groupby(["quiz_id", "quiz_title"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)
print(top_quizzes)

print("\nTop 15 authors by question count:")
print(df["author_name"].value_counts().head(15))

Top 15 quizzes by question count:
quiz_id                   quiz_title                                      
6579826da5cb62cc0dc29451  Nombres Complexes                                   31
66aa51fe401642095c2c0fb2  صِيَغُ الأفعَالِ ودَلَالَتُهَا عَلَى الزّمَانِ      24
659d056d7741e674aea83eb7  Suites réelles ok                                   22
6668755442a9c092cf9fcba4  2مَعَانِي حُرُوفِ الجَرِّ                           22
65cb43b1379dd0177d2c9e35  Intégrales                                          22
66ae0e3b401642095c2e4867  الانقِضاءُ وَعَدَمُ الانقِضَاءِ                     21
683834047519b6796a6a346d  Equations de droites, de plans et de sphères        21
6838425c7519b6796a6a5f17  Equations de droites, de plans et de sphères (2)    21
66af5908401642095c2ef02c  الوُجُوبُ والإمكَانُ                                21
65cddd51c3a6a8d0f6de4e8c  Probabilités                                        20
65cdd567c3a6a8d0f6de3ea8  Arithmétique                                        20


## 12. Sample rows per target level

One example question per target level, to eyeball quality.

In [63]:
for lv in TARGET_LEVELS:
    matches = df[df["levels"].map(lambda lvs: lv in lvs)]
    print(f"\n=== [{LEVEL_TO_STAGE[lv]}] {lv} — {len(matches)} rows ===")
    if matches.empty:
        continue
    sample = matches.iloc[0]
    print(f"  quiz_title : {sample['quiz_title']}")
    print(f"  language   : {sample['language']}")
    print(f"  subjects   : {sample['subjects']}")
    print(f"  question   : {sample['question_text']}")
    print(f"  choices    : {sample['choices_text']}")
    print(f"  correct    : {sample['correct_choices_text']}")


=== [HIGH_SCHOOL] HIGH_SCHOOL_1ST_GRADE — 801 rows ===
  quiz_title : النّظام الغذائي صحّيّ
  language   : ar
  subjects   : []
  question   : ما هو النظام الغذائي الصحي؟
  choices    : ['نظام غذائي متوازن يشمل جميع العناصر الغذائية والمعادن والفيتامينات', 'تناول أطعمة متنوعة تحتوي على العديد من العناصر الغذائية', 'تناول كَميَّة مناسبة من المواد الغذائية المُختلفة', 'كُل ما ذُكِرَ']
  correct    : ['كُل ما ذُكِرَ']

=== [HIGH_SCHOOL] HIGH_SCHOOL_2ND_GRADE_COMPUTER_SCIENCE — 1002 rows ===
  quiz_title : النّظام الغذائي صحّيّ
  language   : ar
  subjects   : []
  question   : ما هو النظام الغذائي الصحي؟
  choices    : ['نظام غذائي متوازن يشمل جميع العناصر الغذائية والمعادن والفيتامينات', 'تناول أطعمة متنوعة تحتوي على العديد من العناصر الغذائية', 'تناول كَميَّة مناسبة من المواد الغذائية المُختلفة', 'كُل ما ذُكِرَ']
  correct    : ['كُل ما ذُكِرَ']

=== [HIGH_SCHOOL] HIGH_SCHOOL_2ND_GRADE_ECONOMICS_AND_SERVICE — 1036 rows ===
  quiz_title : النّظام الغذائي صحّيّ
  language   : ar
  subjec

## 13. Save filtered subset

Persist the slice to `data/interim/levels_subset_<TAG>.jsonl` so other notebooks /
scripts can load it without re-filtering.

In [64]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w") as f:
    for rec in rows:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Wrote {len(rows)} rows to {OUT_PATH}")
print(f"Size: {OUT_PATH.stat().st_size / 1024:.1f} KB")

Wrote 3918 rows to /Users/macmini/Documents/work/softy/quiz-generator/data/interim/levels_subset_HIGH_SCHOOL.jsonl
Size: 4429.9 KB
